---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 12 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


### Menyiapkan SparkSession

Jalankan cell berikut untuk membuat sparksession.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/12 08:50:04 WARN Utils: Your hostname, angger resolves to a loopback address: 127.0.1.1; using 10.90.70.175 instead (on interface wlp3s0)
26/11/12 08:50:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 08:50:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ket

In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 0. Load Dataset ke PySpark DataFrame
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

# 1. StringIndexer
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

df_indexed = indexer_menu.transform(df)
df_indexed = indexer_cabang.transform(df_indexed)
df_indexed = indexer_waktu.transform(df_indexed)

# 2. VectorAssembler
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)


+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.

In [5]:
from pyspark.ml.regression import LinearRegression

# 1. Train/Test Split (75% training, 25% testing, seed=101)
train_data, test_data = df_final.randomSplit([0.75, 0.25], seed=101)

# 2. Inisialisasi dan Latih Model Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr.fit(train_data)

print(f"Jumlah data training : {train_data.count()}")
print(f"Jumlah data testing  : {test_data.count()}")
print("Model utama berhasil dilatih.")


Jumlah data training : 1839
Jumlah data testing  : 661
Model utama berhasil dilatih.


**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [6]:
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Prediksi pada data testing
predictions = model_utama.transform(test_data)

# 2. Evaluasi RMSE dan R2
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE Model Utama : {rmse:,.2f}")
print(f"R² Model Utama   : {r2:.4f}")

# Interpretasi:
# Model ini memiliki performa yang sangat baik untuk memprediksi omzet karena nilai R² mendekati 1 (sangat tinggi).
# Hal ini wajar karena total_penjualan secara langsung dibentuk oleh (qty * harga_satuan) ditambah noise acak kecil.
# Nilai RMSE menunjukkan rata-rata deviasi/error prediksi terhadap nilai penjualan aktual masih berada di kisaran noise tersebut.


RMSE Model Utama : 10,900.80
R² Model Utama   : 0.9107


**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [7]:
# 1. Assembler hanya menggunakan fitur qty dan harga_satuan
assembler_simple = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_simple"
)

df_simple = assembler_simple.transform(df)

# 2. Train/Test Split dengan seed yang sama (101)
train_simple, test_simple = df_simple.randomSplit([0.75, 0.25], seed=101)

# 3. Latih model kedua
lr_simple = LinearRegression(featuresCol="features_simple", labelCol="total_penjualan")
model_simple = lr_simple.fit(train_simple)

# 4. Evaluasi R2 Model Kedua
pred_simple = model_simple.transform(test_simple)
r2_simple = evaluator_r2.evaluate(pred_simple)
rmse_simple = evaluator_rmse.evaluate(pred_simple)

print(f"R² Model Utama (Semua Fitur)        : {r2:.4f} (RMSE: {rmse:,.2f})")
print(f"R² Model Kedua (Hanya Qty & Harga)  : {r2_simple:.4f} (RMSE: {rmse_simple:,.2f})")

# Kesimpulan:
# Menambahkan informasi kategori (menu, cabang, dan waktu) tidak meningkatkan akurasi prediksi secara signifikan.
# Nilai R² dan RMSE kedua model hampir identik, membuktikan bahwa fitur utama yang menentukan total_penjualan 
# hanyalah qty dan harga_satuan.


R² Model Utama (Semua Fitur)        : 0.9107 (RMSE: 10,900.80)
R² Model Kedua (Hanya Qty & Harga)  : 0.9108 (RMSE: 10,896.03)


**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

In [8]:
# 1. Buat DataFrame 3 skenario pesanan baru (data dummy)
data_baru = [
    ("Kopi Susu", "Yogyakarta", "Pagi", 3, 18000),
    ("Cappuccino", "Semarang", "Sore", 2, 25000),
    ("Americano", "Magelang", "Malam", 4, 20000)
]
columns = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_baru = spark.createDataFrame(data_baru, columns)

# 2. Transformasi menggunakan model StringIndexer yang sudah di-fit sebelumnya
df_baru_indexed = indexer_menu.transform(df_baru)
df_baru_indexed = indexer_cabang.transform(df_baru_indexed)
df_baru_indexed = indexer_waktu.transform(df_baru_indexed)

# 3. Transformasi ke vector features menggunakan assembler model utama
df_baru_final = assembler.transform(df_baru_indexed)

# 4. Prediksi total_penjualan
hasil_prediksi = model_utama.transform(df_baru_final)

# Tampilkan hasil prediksi
hasil_prediksi.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan",
    col("prediction").alias("prediksi_total_penjualan")
).show(truncate=False)


+----------+----------+-----------+---+------------+------------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediksi_total_penjualan|
+----------+----------+-----------+---+------------+------------------------+
|Kopi Susu |Yogyakarta|Pagi       |3  |18000       |53805.4580437407        |
|Cappuccino|Semarang  |Sore       |2  |25000       |54094.22750443933       |
|Americano |Magelang  |Malam      |4  |20000       |82081.53157778077       |
+----------+----------+-----------+---+------------+------------------------+

